# Real-ESRGAN Official — ทดสอบใน Colab

In [ ]:
# @title 1. ติดตั้ง
!pip install realesrgan opencv-python --quiet
print("Done")

In [ ]:
# @title 2. อัปโหลดรูป
from google.colab import files
uploaded = files.upload()
input_path = list(uploaded.keys())[0] if uploaded else None
print(f"Input: {input_path}")

In [ ]:
# @title 3. รัน official RealESRGAN
import os, time, urllib.request
import cv2
import torch

from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer

model_path = "RealESRGAN_x2plus.pth"
if not os.path.exists(model_path):
    url = "https://huggingface.co/nateraw/real-esrgan/resolve/main/RealESRGAN_x2plus.pth"
    print("Downloading weights...")
    urllib.request.urlretrieve(url, model_path)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=2)
upsampler = RealESRGANer(
    scale=2,
    model_path=model_path,
    model=model,
    tile=400,
    tile_pad=10,
    pre_pad=0,
    half=False,
    device=device,
)

img = cv2.imread(input_path, cv2.IMREAD_COLOR)
h, w = img.shape[:2]
print(f"Input: {w}x{h}  |  Device: {device}")

t0 = time.time()
output, _ = upsampler.enhance(img, outscale=2)
t = time.time() - t0

oh, ow = output.shape[:2]
print(f"Output: {ow}x{oh}  |  Time: {t:.1f}s")
cv2.imwrite("output_official.png", output)
print("Saved: output_official.png")

In [ ]:
# @title 4. แสดงผล + ดาวน์โหลด
from google.colab.patches import cv2_imshow
from google.colab import files

img = cv2.imread("output_official.png")
cv2_imshow(img)
files.download("output_official.png")